In [2]:
import joblib

In [1]:
import numpy as np
import pandas as pd
import joblib

new_raw_df = pd.read_csv("new_unseen_units.csv")

sensor_cols = ["vibration_mm_s", "coolant_pressure_kpa", "compressor_temp_c", "cargo_temp_deviation_c"]
WINDOW = 7

new_raw_df = new_raw_df.sort_values(["unit_id", "day"]).reset_index(drop=True)

feature_frames = []
for unit_id, g in new_raw_df.groupby("unit_id"):
    g = g.copy()
    for col in sensor_cols:
        g[f"{col}_roll_mean"] = g[col].rolling(WINDOW, min_periods=1).mean()
        g[f"{col}_roll_std"] = g[col].rolling(WINDOW, min_periods=1).std().fillna(0)
        g[f"{col}_trend"] = g[col] - g[col].shift(WINDOW).fillna(g[col])
    feature_frames.append(g)

new_feat_df = pd.concat(feature_frames, ignore_index=True)

feature_cols = [c for c in new_feat_df.columns if c not in ["unit_id", "day"]]

rf = joblib.load("rf_model.joblib")
new_X = new_feat_df[feature_cols]

new_feat_df["predicted_probability"] = rf.predict_proba(new_X)[:, 1]
new_feat_df["predicted_label"] = rf.predict(new_X)

new_feat_df.sort_values("predicted_probability", ascending=False).head(15)

,unit_id,day,vibration_mm_s,coolant_pressure_kpa,compressor_temp_c,cargo_temp_deviation_c,vibration_mm_s_roll_mean,vibration_mm_s_roll_std,vibration_mm_s_trend,coolant_pressure_kpa_roll_mean,coolant_pressure_kpa_roll_std,coolant_pressure_kpa_trend,compressor_temp_c_roll_mean,compressor_temp_c_roll_std,compressor_temp_c_trend,cargo_temp_deviation_c_roll_mean,cargo_temp_deviation_c_roll_std,cargo_temp_deviation_c_trend,predicted_probability,predicted_label
196,1004,16,4.206,121.11,74.46,1.489,4.415571,0.466240,0.010,122.505714,4.207913,-3.91,78.257143,4.358902,-1.03,2.155429,0.713721,-0.516,0.998007,1
201,1004,21,4.424,116.09,78.47,2.778,4.539571,0.446947,-0.556,122.934286,5.210268,-5.14,76.061429,4.323638,7.48,2.041000,0.533072,-0.121,0.997959,1
195,1004,15,3.834,125.93,76.64,2.266,4.414143,0.467004,-0.219,123.064286,4.251054,0.56,78.404286,4.224677,1.30,2.229143,0.657881,0.934,0.997905,1
463,1009,35,4.533,124.97,76.04,2.609,3.871286,0.447074,0.085,122.535714,4.585135,9.62,81.234286,6.105402,-1.54,2.504714,0.509832,0.486,0.997867,1
460,1009,32,3.810,116.61,84.75,2.952,4.053286,0.371510,0.128,122.060000,5.685174,-7.87,80.305714,6.602789,6.86,2.405571,0.488827,1.314,0.997830,1
458,1009,30,4.197,119.89,87.51,2.759,4.182143,0.271362,1.023,121.280000,4.090069,-5.35,78.682857,5.798574,6.40,2.091429,0.505430,0.591,0.997654,1
200,1004,20,4.409,128.14,81.20,2.463,4.619000,0.471704,0.060,123.668571,4.381162,0.93,74.992857,4.547680,1.13,2.058286,0.562123,0.249,0.994594,1
424,1008,42,4.582,114.10,83.89,2.421,4.292857,0.248682,1.014,110.651429,2.829702,-2.35,86.134286,3.689774,4.99,1.938429,0.575069,0.561,0.994013,1
422,1008,40,4.578,114.19,85.63,2.371,4.170857,0.342055,0.699,111.610000,3.399667,0.19,84.728571,4.096594,4.09,1.919000,0.572257,0.943,0.993894,1
462,1009,34,3.946,121.81,85.95,2.400,3.859143,0.426800,-0.241,121.161429,5.141804,-5.61,81.454286,5.911711,14.59,2.435286,0.526096,0.682,0.991147,1


In [2]:
# Check: does predicted_probability actually vary across all rows, or is everything just 1?
print(new_feat_df["predicted_probability"].describe())
print(new_feat_df["predicted_label"].value_counts())

count    525.000000
mean       0.079961
std        0.248155
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        0.998007
Name: predicted_probability, dtype: float64
predicted_label
0    483
1     42
Name: count, dtype: int64
